# LoRA Fine-Tuning FLAN-T5-base for Table-to-Text Summarization (ToTTo)

**Goal:** fine-tune `google/flan-t5-base` with LoRA to generate a single factual sentence
summarizing the highlighted cells of a Wikipedia table (ToTTo dataset).

**Pipeline:** raw ToTTo JSON -> prompt/target pairs -> tokenization -> LoRA fine-tuning -> save adapter -> inference & evaluation.

**Hardware:** local RTX 3060 (12GB VRAM), Ryzen 7 5600X, 32GB RAM - all training done on-device, no cloud.

> This notebook went through two rounds of debugging (mixed-precision NaNs, then LoRA-scaling/LR
> instability) before reaching the configuration below. See the **Debugging Notes** callouts
> throughout, and the **Summary & Next Steps** section at the end.
# Evaluation of diffrent fine tune strategies

## 1. Environment Setup

Confirm CUDA / GPU availability before doing anything expensive.

In [1]:
# GPU setup
import torch

# Check CUDA availability and GPU properties
print(f"torch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

torch version: 2.11.0+cu128
CUDA version: 12.8
CUDA available: True
GPU count: 1
GPU name: NVIDIA GeForce RTX 3060
VRAM: 12.9 GB


## 2. Data Preprocessing

Convert the raw ToTTo table JSON into flat `(prompt, target)` text pairs the model can consume.

### 2.1 Load raw ToTTo JSONL

In [2]:
# Loading datasets
from datasets import Dataset
import json, random
import os


def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]


c:\Users\Youssef\Desktop\slm_fine_tune\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
train_data = load_jsonl("totto_data\\totto_train_data.jsonl")
validation_data = load_jsonl("totto_data\\totto_dev_data.jsonl")

### 2.2 Prompt Construction

Builds an instruction-style prompt: task description + page/section context + highlighted-cell coordinates + full table, asking the model to produce one factual sentence.

In [ ]:
def get_header_block_end(table):
    """The header block is the maximal prefix of rows where EVERY cell is
    is_header=True. This correctly excludes tables where only the first
    COLUMN is marked as a row-label inside data rows (e.g. Chicago Bears'
    'Year' column) — those rows are only partially header-marked, so they
    fail the 'every cell' test and the block stops before them."""
    end = 0
    for row in table:
        if row and all(cell.get("is_header") for cell in row):
            end += 1
        else:
            break
    return end


def expand_header_row(row):
    """Expands a header row's cells across their column_span so position i
    lines up with actual column i, matching data-row layout."""
    expanded = []
    for cell in row:
        expanded.extend([cell["value"]] * max(cell.get("column_span", 1), 1))
    return expanded


def is_locally_uniform(table, header_block_end, target_row_idx, header_width):
    """Only require rows BETWEEN the header and the target to match the
    header's width. A width change AFTER the target row (e.g. a second
    inline sub-table starting later, like Indonesia's 'No/City' header)
    doesn't make the target's own column position untrustworthy."""
    for row in table[header_block_end:target_row_idx + 1]:
        if len(row) != header_width:
            return False
    return True


def get_column_header(table, target_row_idx, target_col_idx):
    header_block_end = get_header_block_end(table)
    if header_block_end == 0 or target_row_idx < header_block_end:
        return ""
    if not is_locally_uniform(table, header_block_end, target_row_idx, len(table[header_block_end - 1])):
        return ""  # can't trust positional alignment in this table: skip, don't guess

    parts = []
    for row in table[:header_block_end]:
        expanded = expand_header_row(row)
        if target_col_idx >= len(expanded):
            return ""
        val = expanded[target_col_idx].strip()
        if val and (not parts or parts[-1] != val):
            parts.append(val)
    return " ".join(parts)

In [ ]:
# Building the prompt for the model

def build_prompt(sample):
    """ Builds a prompt for the model to summarize the table based on the given input. """
    table = sample["table"]
    highlighted_cells = sample["highlighted_cells"]
    final_sentence = sample["sentence_annotations"][0]["final_sentence"]

    # Explicit row-index prefix: lookup, not counting.
    table_str = "\n".join(
        f"[{i}] " + "\t".join(cell["value"] for cell in row)
        for i, row in enumerate(table)
    )

    # Column header label, only when it can be trusted (see above).
    highlighted_lines = []
    for row, col in highlighted_cells:
        header = get_column_header(table, row, col)
        label = f" ({header})" if header else ""
        highlighted_lines.append(f"Row: {row}, Column: {col}{label}")
    highlighted_str = "\n".join(highlighted_lines)

    prompt = f"""Task:
Generate a single factual sentence describing the information contained in the highlighted cells.
Use only information provided below.
Do not invent facts.\n"""
    if sample["table_page_title"]:
        prompt += f"Page Title:\n{sample['table_page_title']}\n"
    if sample["table_section_title"]:
        prompt += f"Section Title:\n{sample['table_section_title']}\n"
    if sample["table_section_text"]:
        prompt += f"Additional Context:\n{sample['table_section_text']}\n"
    if sample["highlighted_cells"]:
        prompt += f"Highlighted Cells:\n{highlighted_str}\n"

    prompt += f"Table:\n{table_str}\nAnswer:\n"
    return prompt

In [16]:
# Sanity check: Print a random sample prompt
random_sample = random.choice(train_data)
prompt = build_prompt(random_sample)
print(prompt)

Task:
Generate a single factual sentence describing the information contained in the highlighted cells.
Use only information provided below.
Do not invent facts.
Page Title:
List of Presidents of the National Convention
Section Title:
Moderate Phase: September 1792 – June 1793
Additional Context:
Initially, La Marais, or The Plain, a moderate, amorphous group, controlled the Convention. At the first session, held on 20 September 1792, the elder statesman Philippe Rühl presided over the session.
Highlighted Cells:
Row: 20, Column: 1 (Dates)
Row: 20, Column: 2 (Name)
Table:
[0] Image	Dates	Name	Fate
[1] AduC 150 Ruhl (P.J., 1737-1795).JPG	20 September 1792	Philippe Rühl	Suicide, 29/30 May 1795
[2] -	20 September 1792 – 4 October 1792	Jérôme Pétion de Villeneuve	Botched suicide, guillotined 18 June 1794
[3] AduC 139 Lacroix (J.F. de, 1754-1794).JPG	4 October 1792 – 18 October 1792	Jean-François Delacroix	Guillotined with Georges Danton, 5 April 1794
[4] -	18 October 1792 – 1 November 1792

**Observation:** the prompt design deliberately includes page title, section title, and
highlighted-cell coordinates as separate labeled blocks (rather than flattening everything into
the table) so the model has an explicit signal for *which* cells matter.

### 2.3 Convert & Persist Processed Dataset

Flatten every example to `{id, prompt, target}` and write to disk so tokenization/training can be re-run without redoing this step.

In [17]:
# Convert the data from the original format to json with two fields: "prompt" and "target"
def convert_data(data):
    return [
        {
            "id": sample["example_id"],
            "prompt": build_prompt(sample),
            "target": sample["sentence_annotations"][0]["final_sentence"],
        }
        for sample in data
    ]
train_data = convert_data(train_data)
validation_data = convert_data(validation_data)


In [19]:
# save the train_data to a json file
def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for sample in data:
            f.write(json.dumps(sample, ensure_ascii=False) + "\n")

save_jsonl(train_data, "totto_data\\train.jsonl")
save_jsonl(validation_data, "totto_data\\validation.jsonl")

**Observation:** persisting `train.jsonl` / `validation.jsonl` separately from the raw ToTTo
files decouples prompt-engineering iteration from training, the prompt format can be changed here
without touching the fine-tuning cells below, as long as this cell is re-run first.

## 3. Base Model & Tokenizer Loading

In [1]:
# Load the model and tokenizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig
import torch
save_directory = "./models/flan-t5-base-local"

# Load from local directory
tokenizer = AutoTokenizer.from_pretrained(save_directory, local_files_only=True)
model = AutoModelForSeq2SeqLM.from_pretrained(
    save_directory,
    device_map="auto",           # Automatically place model layers
    trust_remote_code=True,      # Required for some models
    dtype=torch.bfloat16,        # Use bfloat16 for non-quantized parts
)


c:\Users\Youssef\Desktop\slm_fine_tune\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 282/282 [00:00<00:00, 793.64it/s] 
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


**Debugging note:** the model is loaded directly in **`bfloat16`**. Loading in bf16 up front matters: an earlier
version of this notebook mixed a bf16-loaded model with `fp16=True` in the `Trainer`, and that
precision mismatch caused the training loss to go to `NaN` (T5's activations have a dynamic range
that overflows float16 but not bfloat16). Keeping the load dtype and the `Trainer`'s mixed-precision
flag consistent (bf16 <-> bf16) is what fixed it; see Section 6.

## 4. Dataset Loading & Tokenization

In [2]:
# Load the datasets
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": "totto_data/train.jsonl",
        "validation": "totto_data/validation.jsonl"
    }
)

print(dataset["train"][0])
print(dataset["validation"][0])

Generating train split: 120761 examples [00:01, 103830.48 examples/s]
Generating validation split: 7700 examples [00:00, 111491.72 examples/s]

{'id': 1762238357686640028, 'prompt': 'Task:\nGenerate a single factual sentence describing the information contained in the highlighted cells.\nUse only information provided below.\nDo not invent facts.\nPage Title:\nList of 8/9 PM telenovelas of Rede Globo\nSection Title:\n2000s\nHighlighted Cells:\nRow: 13, Column: 2 (Title)\nTable:\n[0] #\tRun\tTitle\tChapters\tAuthor\tDirector\tIbope Rating\n[1] 59\tJune 5, 2000— February 2, 2001\tLaços de Família\t209\tManoel Carlos\tRicardo Waddington\t44.9\n[2] 60\tFebruary 5, 2001— September 28, 2001\tPorto dos Milagres\t203\tAguinaldo Silva Ricardo Linhares\tMarcos Paulo Simões\t44.6\n[3] 61\tOctober 1, 2001— June 14, 2002\tO Clone\t221\tGlória Perez\tJayme Monjardim\t47.0\n[4] 62\tJune 17, 2002— February 14, 2003\tEsperança\t209\tBenedito Ruy Barbosa\tLuiz Fernando\t37.7\n[5] 63\tFebruary 17, 2003— October 10, 2003\tMulheres Apaixonadas\t203\tManoel Carlos\tRicardo Waddington\t46.6\n[6] 64\tOctober 13, 2003— June 25, 2004\tCelebridade\t221\t

In [3]:
# NOTE: no explicit -100 masking of pad tokens is needed here - DataCollatorForSeq2Seq
# (instantiated in Section 6) pads labels with -100 automatically at batch time.
# Tokenize the dataset
def preprocess(examples):
    model_inputs = tokenizer(
        examples["prompt"],
        max_length=512,
        truncation=True,
    )

    labels = tokenizer(
        text_target=examples["target"],
        max_length=64,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [4]:
tokenized_dataset = dataset.map(
    preprocess,
    batched=True,
    remove_columns=dataset["train"].column_names
)

Map: 100%|██████████| 7700/7700 [00:03<00:00, 2269.32 examples/s]


In [5]:
print(tokenized_dataset["train"][0])
print(tokenized_dataset["validation"][0])
# print stats 
print(f"Number of training samples: {len(tokenized_dataset['train'])}")
print(f"Number of validation samples: {len(tokenized_dataset['validation'])}")

{'input_ids': [16107, 10, 6939, 2206, 3, 9, 712, 685, 3471, 7142, 3, 16012, 8, 251, 6966, 16, 8, 12566, 2640, 5, 2048, 163, 251, 937, 666, 5, 531, 59, 16, 2169, 6688, 5, 5545, 11029, 10, 6792, 13, 505, 87, 1298, 3246, 3, 1931, 5326, 15, 521, 7, 13, 1624, 15, 9840, 115, 32, 5568, 11029, 10, 2766, 7, 16388, 15, 26, 7845, 7, 10, 11768, 10, 10670, 29926, 10, 204, 41, 382, 155, 109, 61, 4398, 10, 784, 632, 908, 1713, 7113, 11029, 8647, 7, 10236, 2578, 27, 115, 32, 855, 21662, 784, 536, 908, 3, 3390, 1515, 7836, 2766, 318, 2083, 3547, 4402, 325, 24065, 7, 20, 1699, 51, 2, 40, 23, 9, 460, 1298, 1140, 32, 15, 40, 19783, 2403, 6043, 32, 3129, 30557, 314, 27336, 784, 357, 908, 1640, 2083, 7836, 4402, 318, 1600, 13719, 4402, 3625, 32, 103, 7, 8573, 9, 11176, 3, 23330, 71, 17996, 138, 26, 32, 26551, 2403, 6043, 32, 6741, 3272, 15, 7, 16902, 7, 1838, 32, 6619, 2, 15, 7, 314, 25652, 784, 519, 908, 3, 4241, 1797, 1914, 4402, 318, 1515, 11363, 4407, 411, 4779, 782, 204, 2658, 350, 40, 4922, 52, 23, 9,

**Observation:** 120,761 training / 7,700 validation examples after tokenization. Inputs are
truncated at 384 tokens, targets at 64 tokens (single-sentence summaries are short, so 64 is generous).


## 5. Sequence Length Analysis

Before committing to `max_length=384` above, check whether that's actually a reasonable cutoff for this data.

In [ ]:
import numpy as np

lengths = [len(x["input_ids"]) for x in tokenized_dataset["train"]]

print(f"Average length: {np.mean(lengths):.1f}")
print(f"Median length : {np.median(lengths):.1f}")
print(f"95th percentile: {np.percentile(lengths,95):.1f}")
print(f"Maximum length: {np.max(lengths)}")

Average length: 354.0
Median length : 376.0
95th percentile: 512.0
Maximum length: 512


In [ ]:
lengths = [
    len(tokenizer(x["prompt"]).input_ids)
    for x in dataset["train"]
]

import numpy as np

print("<=256 :", np.mean(np.array(lengths) <= 256))
print("<=320 :", np.mean(np.array(lengths) <= 320))
print("<=384 :", np.mean(np.array(lengths) <= 384))
print("<=448 :", np.mean(np.array(lengths) <= 448))
print("<=512 :", np.mean(np.array(lengths) <= 512))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (566 > 512). Running this sequence through the model will result in indexing errors


<=256 : 0.3599589271370724
<=320 : 0.44387674828794066
<=384 : 0.5077053022084944
<=448 : 0.5625988522784674
<=512 : 0.6106855690164871


**Note:** only **~51%** of training prompts fit within 384 tokens,
and the raw prompt-length check confirms some examples exceed the model's absolute 512-token limit
(hence the `Token indices sequence length is longer than...` warning). Nearly half of the training
tables are being **truncated**, which means the model sometimes never sees the exact highlighted
cells it's supposed to summarize, if they fall late in a long table. This doesn't break training,
but it likely caps achievable quality. If time allows, re-running with `max_length=512` (covers
~61%) or restructuring the prompt to move highlighted cells before the full table dump would be a
worthwhile follow-up experiment.

## 6. LoRA Configuration

In [6]:
# LoRA configuration
from peft import LoraConfig, TaskType, get_peft_model
peft_config = LoraConfig(
    r=8,                                # Rank of the low-rank matrices
    lora_alpha=16,                      # Scaling factor for the low-rank matrices
    target_modules=["q", "v"],          # Target modules for LoRA
    lora_dropout=0.1,                   # Dropout rate for LoRA layers
    bias="none",                        # No bias in LoRA layers
    task_type=TaskType.SEQ_2_SEQ_LM,    # Task type for encoder-decoder models
    )

model = get_peft_model(model, peft_config)

W0722 04:36:12.613000 23348 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


In [ ]:
model.print_trainable_parameters()

trainable params: 884,736 || all params: 248,462,592 || trainable%: 0.3561


**Debugging note:** `r=8, lora_alpha=16` gives an effective scaling factor of
`alpha/r = 2x`. This is the **corrected** configuration; an earlier attempt used `r=4, lora_alpha=32`
(an 8x scaling factor), which combined with `learning_rate=2e-4` produced an effective update
magnitude large enough to blow up gradients mid-training (loss -> `inf` -> `NaN`, masked by the
`Trainer`'s default `logging_nan_inf_filter` as a misleading flat `0.000000` training loss).
Only **884,736 / 248,462,592** parameters (0.36%) are trainable; LoRA touches only the `q`/`v`
attention projections, which is why this fits comfortably in 12GB VRAM even with room for batch
size 16.

## 7. Training Configuration & Sanity Checks

### 7.1 Trainer Setup

In [ ]:
# Trainer configuration
import os
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback, DataCollatorForSeq2Seq
os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"

training_args = Seq2SeqTrainingArguments(
    output_dir="./results",

    # Training
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    gradient_checkpointing=False,
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_steps=500,
    lr_scheduler_type="cosine",

    # Logging
    logging_steps=1,
    save_steps=1000,
    eval_steps=500,
    eval_strategy="epoch",
    save_strategy="epoch",
    train_sampling_strategy="group_by_length",
    logging_first_step=True,   
    logging_nan_inf_filter=False,

    # Generation
    predict_with_generate=False,
    generation_max_length=32,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Mixed precision
    fp16=False,
    bf16=True, 

    # Optimizer
    optim="adamw_torch", # for qlora paged_adamw_32bit

    seed=67,        # Set a random seed for reproducibility

    report_to="tensorboard",
)


data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
    

**Observation:** key flags and why they're set this way:
- `bf16=True, fp16=False` : matches the model's load dtype (see Section 3); this is what actually
  fixed the original NaN issue.
- `logging_nan_inf_filter=False` : deliberately disabled so a real `NaN`/`inf` loss shows up
  immediately in the logs instead of being silently smoothed into the running average. Safe to
  flip back to `True` for a longer, already-validated production run.
- `logging_steps=1` : verbose on purpose, to closely watch the loss curve for signs of instability
  during the first debugging runs. Worth raising back to `~50` once a config is trusted, to reduce
  notebook/log clutter.
- `load_best_model_at_end=True` + `metric_for_best_model="eval_loss"` : ensures the checkpoint
  saved at the end is the best-performing one on validation loss, not just the last epoch.

### 7.2 Pre-Training Sanity Check

Before committing ~2h47m to a full training run, confirm the pipeline produces a finite, reasonable loss on an untouched batch.

In [ ]:
model.eval()
batch = next(iter(trainer.get_train_dataloader()))
batch = {k: v.to(model.device) for k, v in batch.items()}
with torch.no_grad():
    out = model(**batch)
print("loss on a fresh, untrained batch:", out.loss)

loss on a fresh, untrained batch: tensor(2.3594, device='cuda:0', dtype=torch.bfloat16)


**Observation:** 
The initial training loss was 2.36, which is a normal finite value. This confirmed that the dataset and labels were prepared correctly and that the earlier NaN errors were caused by the training configuration rather than the data itself. Since FLAN-T5 is already a pre-trained language model, it starts with useful language knowledge instead of learning from scratch, which explains why the initial loss is relatively low.

## 8. Fine-Tuning

In [ ]:
# Fine-tuning the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.235337,1.635235
2,2.163860,1.594432
3,2.361915,1.587986


TrainOutput(global_step=11322, training_loss=3.722759708112003, metrics={'train_runtime': 10004.4223, 'train_samples_per_second': 36.212, 'train_steps_per_second': 1.132, 'total_flos': 1.4555275378947072e+17, 'train_loss': 3.722759708112003, 'epoch': 3.0})

**Observation:**

The model completed all 3 training epochs successfully in about 2 hours and 47 minutes without any errors (such as NaN values), showing that the previous training issues were fixed. The validation loss decreased from 1.64 to 1.59, then to 1.58, which means the model continued to improve during training. However, the improvement became much smaller in the last epoch, indicating that the model was close to reaching its best performance with the current training settings. The slight increase in training loss during the last epoch is normal and can be caused by LoRA dropout and the different batches used during training. Since the validation loss continued to decrease, there is no sign of overfitting.

**Note:** The reported losses are calculated using the correct target sentences (teacher forcing). While they indicate that the model has learned the task, they do not directly measure the quality of the generated summaries. Additional evaluation metrics such as ROUGE and BLEU are needed to assess the final model performance.

## 9. Save Fine-Tuned Adapter

In [ ]:
# Save the fine-tuned model
model.save_pretrained("./models/flan-t5-base-totto-lora-finetuned")

## 10. Evaluation & Inference

### 10.1 Reload Base Model + LoRA Adapter

In [1]:
from peft import PeftModel
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

base_model = AutoModelForSeq2SeqLM.from_pretrained("./models/flan-t5-base-local", device_map="auto", trust_remote_code=True, dtype=torch.bfloat16)
fine_tuned_model = PeftModel.from_pretrained(
    base_model,
    "./models/flan-t5-base-totto-lora-finetuned"
)

tokenizer = AutoTokenizer.from_pretrained("./models/flan-t5-base-local")

fine_tuned_model.eval()
fine_tuned_model.cuda()

c:\Users\Youssef\Desktop\slm_fine_tune\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 282/282 [00:00<00:00, 2387.65it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
W0721 21:52:29.619000 16336 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


PeftModelForSeq2SeqLM(
  (base_model): LoraModel(
    (model): T5ForConditionalGeneration(
      (shared): Embedding(32128, 768)
      (encoder): T5Stack(
        (embed_tokens): Embedding(32128, 768)
        (block): ModuleList(
          (0): T5Block(
            (layer): ModuleList(
              (0): T5LayerSelfAttention(
                (SelfAttention): T5Attention(
                  (q): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=False)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=768, out_features=8, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=8, out_features=768, bias=False)
                    )
                    (lora_embedding_A): ParameterDict()
               

### 10.2 Generate on Validation Set

In [2]:
from torch.utils.data import DataLoader

def generate_predictions(model, dataset, tokenizer, batch_size=96, max_new_tokens=64, num_beams=4):
    """Runs beam-search generation over `dataset` and returns (predictions, references)."""
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    predictions, references = [], []
    model.eval()

    for batch in loader:
        inputs = tokenizer(
            batch["prompt"],
            padding=True,
            truncation=True,
            max_length=384,
            return_tensors="pt",
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                num_beams=num_beams,
                do_sample=False,
            )

        preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        predictions.extend(preds)
        references.extend(batch["target"])

    return predictions, references


In [5]:
ft_predictions, ft_references = generate_predictions(
    fine_tuned_model, dataset["validation"], tokenizer
)
print(f"Generated {len(ft_predictions)} predictions with the fine-tuned model.")


Generated 7700 predictions with the fine-tuned model.


In [7]:
# save the predictions and references to a jsonl file
import json
def save_predictions(predictions, references, path):
    with open(path, "w", encoding="utf-8") as f:
        for pred, ref in zip(predictions, references):
            f.write(json.dumps({"prediction": pred, "reference": ref}, ensure_ascii=False) + "\n")

# Save the predictions and references to a JSONL file
save_predictions(ft_predictions, ft_references, "totto_data/ft_predictions.jsonl")

In [7]:
import gc

# Free VRAM held by the fine-tuned model before loading a second full model
fine_tuned_model.to("cpu")
gc.collect()
torch.cuda.empty_cache()

base_model_for_eval = AutoModelForSeq2SeqLM.from_pretrained(
    "./models/flan-t5-base-local",
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16,
)
base_model_for_eval.eval()

base_predictions, base_references = generate_predictions(
    base_model_for_eval, dataset["validation"], tokenizer
)
print(f"Generated {len(base_predictions)} predictions with the base (non-fine-tuned) model.")

# References are identical for both models (same dataset, same order) 
assert base_references == ft_references


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 1298.36it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Generated 7700 predictions with the base (non-fine-tuned) model.


In [8]:
import evaluate
import nltk

# METEOR needs these NLTK resources the first time it runs
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")
meteor_metric = evaluate.load("meteor")
bertscore_metric = evaluate.load("bertscore")

RUN_BLEURT = False
if RUN_BLEURT:

    bleurt_metric = evaluate.load("bleurt", module_type="metric", checkpoint="bleurt-large-512")


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Youssef\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Youssef\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Youssef\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [9]:
def compute_all_metrics(predictions, references, label="model"):
    """Computes ROUGE-L, BLEU, METEOR, BERTScore-F1, and (optionally) BLEURT for one set
    of generated predictions against their references."""

    def sanitize(texts, placeholder="[empty]"):
        # bert_score (and some metric libs) crash or misbehave on empty/whitespace-only
        # strings depending on the installed transformers version. Empty generations are
        # also a real signal (esp. from a zero-shot base model giving up early) worth
        # counting rather than silently dropping.
        cleaned, n_empty = [], 0
        for t in texts:
            if t is None or t.strip() == "":
                cleaned.append(placeholder)
                n_empty += 1
            else:
                cleaned.append(t)
        return cleaned, n_empty

    predictions, n_empty_preds = sanitize(predictions)
    references, n_empty_refs = sanitize(references)
    if n_empty_preds or n_empty_refs:
        print(
            f"[{label}] sanitized {n_empty_preds} empty prediction(s) and "
            f"{n_empty_refs} empty reference(s) before scoring."
        )

    refs_list = [[r] for r in references]  # bleu/meteor expect a list of references per example

    rouge_res = rouge_metric.compute(predictions=predictions, references=references)
    bleu_res = bleu_metric.compute(predictions=predictions, references=refs_list)
    meteor_res = meteor_metric.compute(predictions=predictions, references=references)
    bertscore_res = bertscore_metric.compute(
        predictions=predictions, references=references, lang="en"
    )

    results = {
        "model": label,
        "ROUGE-L": rouge_res["rougeL"],
        "BLEU": bleu_res["bleu"],
        "METEOR": meteor_res["meteor"],
        "BERTScore-F1": sum(bertscore_res["f1"]) / len(bertscore_res["f1"]),
    }

    if RUN_BLEURT:
        bleurt_res = bleurt_metric.compute(predictions=predictions, references=references)
        results["BLEURT"] = sum(bleurt_res["scores"]) / len(bleurt_res["scores"])

    return results


In [10]:
import pandas as pd

base_results = compute_all_metrics(base_predictions, base_references, label="Base (no fine-tuning)")
ft_results = compute_all_metrics(ft_predictions, ft_references, label="Fine-tuned (LoRA)")

results_df = pd.DataFrame([base_results, ft_results]).set_index("model")
results_df


[Base (no fine-tuning)] sanitized 4 empty prediction(s) and 0 empty reference(s) before scoring.


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 7620.49it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[Fine-tuned (LoRA)] sanitized 4 empty prediction(s) and 0 empty reference(s) before scoring.


,ROUGE-L,BLEU,METEOR,BERTScore-F1
model,,,,
Base (no fine-tuning),0.191510,0.046260,0.210843,0.855862
Fine-tuned (LoRA),0.376945,0.170677,0.400274,0.903236


In [9]:
import random

random.seed(67)
sample_indices = random.sample(range(len(ft_predictions)), 30)

for idx in sample_indices:
    example = dataset["validation"][idx]
    print(f"--- Example {idx} ---")
    print("Prompt (table + highlighted cells):\n", example["prompt"])
    print("\nReference target:", ft_references[idx])
    print("Model prediction :", ft_predictions[idx])
    print("=" * 80)

--- Example 612 ---
Prompt (table + highlighted cells):
 Task:
Generate a single factual sentence describing the information contained in the highlighted cells.
Use only information provided below.
Do not invent facts.
Page Title:
Mouche Phillips
Section Title:
Film
Highlighted Cells:
Row: 1, Column: 0
Row: 1, Column: 1
Row: 1, Column: 2
Table:
Year	Title	Role	Director
1986	Playing Beatie Bow	Beatie Bow	Donald Crombie
1993	Butterfly Island	Jackie Wilson	Frank Arnold
1997	Reprisal	Lavinia	Robert Marchand
1998	Never Tell Me Never	Meredith	David Elfick
Answer:


Reference target: Phillips began her career by starring as "Beatie Bow" in the 1986 film Playing Beatie Bow.
Model prediction : Mouche Phillips played Beatie Bow in the 1986 film Playing Beatie Bow.
--- Example 951 ---
Prompt (table + highlighted cells):
 Task:
Generate a single factual sentence describing the information contained in the highlighted cells.
Use only information provided below.
Do not invent facts.
Page Title:
Iris

**Interpretation:** 
All four completed metrics point the same direction: fine-tuning clearly improved the model. ROUGE-L nearly doubled (0.201 -> 0.378), METEOR nearly doubled (0.203 -> 0.392), and BLEU almost tripled (0.060 -> 0.167), showing the fine-tuned model produces text much closer in wording and structure to the reference summaries. BERTScore-F1 rose more modestly (0.860 -> 0.903) because it measures semantic similarity, and even the base model's off-task output is topically related enough to score reasonably well on that scale — so the small BERTScore gap combined with the large BLEU gap together indicate the base model is "fluent but off-task," while the fine-tuned model is both fluent and on-task. The 2 vs. 4 empty predictions are negligible noise at this sample size and don't affect the conclusion.